In [2]:
import manim as mn
import random
from manim import *

config.media_width = "75%"
config.verbosity = "WARNING"

print(mn.__version__)

0.20.1


In [18]:
%%manim -qm WetlabProtocol

# https://docs.manim.community/en/stable/reference/manim.utils.color.manim_colors.html

do_full_anim = True

#####################################
##################################### Wetlab protocol
#####################################

def create_bactera():
    a = [-1,-1, 0]
    b = [ 1,-1, 0]
    c = [ 1, 1, 0]
    d = [-1, 1, 0]
    bact_inner_color = GRAY
    ap1 = ArcPolygon(a, b, c, d, fill_opacity=1, color=GRAY, arc_config=[
        {'radius': 10, 'color': bact_inner_color}, 
        {'radius': 1, 'color': bact_inner_color},
        {'radius': 10, 'color': bact_inner_color},
        {'radius': 1, 'color': bact_inner_color}])
    return(ap1)

def create_bact_genome(seed=1):
    bscale = 0.8
    rng = np.random.default_rng(seed=seed)
    list_point = [[
        rng.uniform(-bscale, bscale),
        rng.uniform(-bscale, bscale),
        0] for i in range(15) ]
    ap1 = ArcPolygon(*list_point, color=YELLOW, angle=2)
    return(ap1)


def rand_point_in_circle(rng, radius):
    while True:    
        x=rng.uniform(-radius,radius)
        y=rng.uniform(-radius,radius)
        if x*x+y*y<radius*radius:
            return [x,y,0]

def create_some_dna(seed=1, radius=2, nump=100, color=color): #############################
    #bscale = 1 #0.8/2 #0.8
    cscale = 0.2
    rng = np.random.default_rng(seed=seed)

    def make_line():
        p1=np.array(rand_point_in_circle(rng, radius=radius))
        p2=p1+[
            rng.uniform(-cscale, cscale),
            rng.uniform(-cscale, cscale),
            0
        ]
        return(Line(p1,p2,color=color))
        
    return([make_line() for i in range(nump)])

def create_spc():
    spc_radius = 2
    blue_circle = Circle(fill_color=BLUE, stroke_color=BLACK, fill_opacity=0.5, radius=spc_radius)
    spc_outer = DashedVMobject(Circle(radius=spc_radius, color=WHITE), dashed_ratio=0.5, num_dashes=30)
    #return Group(blue_circle, spc_outer)
    return [blue_circle, spc_outer]

def create_particles(with_color):
    points = [Point(location=[np.random.uniform(-8, 8),  np.random.uniform(-4, 4), 0], color=with_color)
              for i in range(1000)]    
    return Group(*points)
    

class WetlabProtocol(MovingCameraScene):
    def construct(self):

        ######## constants #############
        color_lysis = GREEN
        color_wga = YELLOW
        color_dna = BLUE_B

        ######## create bacteria and spc ###########
        bact_env = create_bactera()
        bact_genome = create_bact_genome()
        bact = Group(bact_env,bact_genome).scale(0.5)
        self.add(bact)
        
        spc = create_spc()

        if do_full_anim:
            self.play(
                AnimationGroup(
                    Create(spc[0],run_time=3),
                    Create(spc[1],run_time=3),
                )
            )   
        else:
            self.add(spc[0])
            self.add(spc[1])        

        ###### show bacteria trapped ######
        if do_full_anim:
            self.play(bact.animate().shift([-1, 0,0]))
            self.play(bact.animate().shift([ 1, 1.2,0]))
            self.play(bact.animate().shift([ 1,-1.2,0]))
            self.play(bact.animate().shift([-1, 0,0]))

        #### show lysis #############
        if do_full_anim:
            part_lysis = create_particles(color_lysis)
            part_lysis.shift([-20,0,0])        
            self.add(part_lysis)
            self.play(part_lysis.animate(run_time=5).shift([20,0,0]))        
            self.play(bact_env.animate(run_time=3).set_opacity(0))
            self.play(part_lysis.animate(run_time=3).shift([20,0,0]))
            #self.wait() #paus not needed
        else:
            bact_env.set_opacity(0)

        #### show WGA #############
        list_dna = create_some_dna(seed=2,radius=1.8,color=color_dna)
        one_dna = Line([0.2,1,0], [0.8,1,0], color=color_dna)        
        if do_full_anim:
            part_wga = create_particles(color_wga)
            part_wga.shift([-20,0,0])        
            self.add(part_wga)
            self.play(part_wga.animate(run_time=5).shift([20,0,0]))        
            self.play(
                *[Create(p) for p in list_dna],
                Create(one_dna)
            )
            self.play(part_wga.animate(run_time=3).shift([20,0,0]))
            self.wait()
        else:
            self.add(*list_dna)
            self.add(one_dna)

        #### show barcode adapter ligation #############
        self.play(
            *[p.animate.set_opacity(0.03) for p in list_dna],
            bact_genome.animate.set_opacity(0.00),
            self.camera.frame.animate.scale(0.1).move_to(one_dna)
        )

        line_bc1 = Line([+0.2,1,0], [+0.1,1,0], color=BLACK)
        line_bc2 = Line([+0.1,1,0], [+0.0,1,0], color=YELLOW)     
        line_bc3 = Line([+0.0,1,0], [-0.1,1,0], color=RED)   
        line_bc4 = Line([-0.1,1,0], [-0.2,1,0], color=GREEN)    
        text_bc1 = Text("A",font_size=5).next_to(line_bc1, [0,-0.1,0])
        text_bc2 = Text("B",font_size=5).next_to(line_bc2, [0,-0.1,0])
        text_bc3 = Text("C",font_size=5).next_to(line_bc3, [0,-0.1,0])
        text_bc4 = Text("D",font_size=5).next_to(line_bc4, [0,-0.1,0])
        text_bc_all = Tex(r"Combinatorial \\barcode",font_size=5).move_to([+0.0,1.1,0])

        if do_full_anim:
            self.play(
                Create(line_bc1),
                Create(text_bc1),
            )
            self.wait()
            self.play(        
                Create(line_bc2),
                Create(text_bc2)
            )
            self.wait()
            self.play(        
                Create(line_bc3),
                Create(text_bc3)
            )
            self.wait()
            self.play(        
                Create(line_bc4),
                Create(text_bc4)
            )
            self.wait()
            self.play(        
                Create(text_bc_all)
            )
            self.wait()
        else:
            self.add(line_bc1)
            self.add(line_bc2)
            self.add(line_bc3)
            self.add(line_bc4)

        ########### open the SPC #############

        def make_ligated_dna():
            line_bc1 = Line([+0.2,1,0], [+0.1,1,0], color=BLACK)
            line_bc2 = Line([+0.1,1,0], [+0.0,1,0], color=YELLOW)     
            line_bc3 = Line([+0.0,1,0], [-0.1,1,0], color=RED)   
            line_bc4 = Line([-0.1,1,0], [-0.2,1,0], color=GREEN)    
            one_dna = Line([0.2,1,0], [0.8,1,0], color=color_dna)        
            #print("a")
            grp = VGroup(
                line_bc1,
                line_bc2,
                line_bc3,
                line_bc4,
                one_dna
            ).shift([0,-1,0])        
            return(grp)

        rng = np.random.default_rng(seed=5)
        for i in range(10):
            more_dna = make_ligated_dna()
            cscale=1
            more_dna.shift([rng.uniform(-cscale, cscale),rng.uniform(-cscale, 0),0])
            self.add(more_dna)
        
        self.play(
            self.camera.frame.animate.scale(6).move_to([0,0,0])
        )
        self.wait()
        self.play(
            *[s.animate.set_opacity(0) for s in spc]
        )
        self.wait()
        #self.play(Restore(self.camera.frame))    

Manim Community v0.20.1

In [55]:
%%manim -qm GiveKMERaNumber

#####################################
##################################### Can turn DNA into a sequence ID
#####################################

class GiveKMERaNumber(Scene):
    def construct(self):
        m0 = MobjectMatrix([
            [Text("A"), Text("00")], 
            [Text("T"), Text("01")],
            [Text("C"), Text("10")],
            [Text("G"), Text("11")]
        ])
        m0.move_to([-4,0,0])

        txt_head1 = Tex(r"We can assign a number \\ to each DNA sequence!")
        txt_head1.move_to([2,2,0])
        self.add(txt_head1)
        
        dna_txt1 = Text(" A")
        dna_txt2 = Text(" A")
        dna_txt3 = Text(" T")
        dna_txt4 = Text(" C")
        grp_a =  VGroup(dna_txt1, dna_txt2, dna_txt3, dna_txt4).arrange()
        grp_a.next_to(txt_head1, DOWN)
        #grp_a.move_to(RIGHT)

        
        txt1 = Text("00")
        txt2 = Text("00")
        txt3 = Text("01")
        txt4 = Text("11")
        grp_b =  VGroup(txt1, txt2, txt3, txt4).arrange()
        grp_b.next_to(grp_a, DOWN)
        
        txt_dec = Text("Decimal: 7") #00000111 1 + 2 + 4 = 7
        txt_dec.next_to(grp_b, DOWN)

        ####### just DNA ##########
        self.wait()
        self.play(
            Create(dna_txt1),
            Create(dna_txt2),
            Create(dna_txt3),
            Create(dna_txt4)
        )
        self.wait()
        self.add(m0)
        self.add(grp_a)
        self.wait()

        ####### Binary conversion ##########
        self.play(Create(txt1))
        self.play(Create(txt2))
        self.play(Create(txt3))
        self.play(Create(txt4))
        self.wait()

        ####### Decimal conversion ##########
        self.play(Create(txt_dec))
        self.wait()

Manim Community v0.20.1

In [667]:
%%manim -qm CountKMERtoHighDim

def get_wx_dna(x):
    return(x*scale_dna_x - 5 )

def get_wx_kmer(x):
    return(x*scale_kmer_x - 5 )

class CountKMERtoHighDim(Scene):
    def construct(self):

        #######
        txt_head1 = Tex(r"Using number for the DNA KMER,\\ we can fill in a vector of KMER counts")
        txt_head1.move_to([0,2,0])
        self.add(txt_head1)
        self.wait()


        ######### constants #################
        scale_dna_x = 0.3
        scale_kmer_x = 0.25
        window_size = scale_dna_x*5
        kmer_size = 5
        kmer_flanks = int(kmer_size/2)
        height_kmer_rect = 0.7
        pos_kmer_y = -2
        pos_kmer_y1 = -height_kmer_rect/2.0
        pos_kmer_y2 = pos_kmer_y + 0.2

        dna_seq = "AACGACAGTACCGA"#CATCGACACTACTACG"
        #into_pos = list(range(len(dna_seq)))
        into_pos = list(range(30))
        random.shuffle(into_pos)
        
        ######### Show count vector #################        

        ## Place DNA
        for i in range(len(dna_seq)):
            text_dna = Text(dna_seq[i], font_size=20).move_to(([get_wx_dna(i), 0, 0]))
            self.add(text_dna)

        self.wait()
        
        ## Place matrix
        num_mat = [0 for i in range(len(into_pos))]
        text_mat = [Text("0", font_size=15).move_to([get_wx_kmer(i), pos_kmer_y, 0]) for i in range(len(num_mat))]
        for t in text_mat:
            self.add(t)
            
        self.wait()
        
        ######### animate the counting #################

        ## Prep rect                 
        pos_dna_rect = [-20, 0, 0] #get_wx_dna(0)
        rect_kmer = Rectangle(width=window_size, height=height_kmer_rect).move_to(pos_dna_rect)
        self.add(rect_kmer)
        #self.play(Create(rect_kmer))
        
        ## Prep arrow  
        pdna_arrow = [get_wx_dna(0), pos_kmer_y1, 0]
        pmat = [get_wx_kmer(into_pos[0]), pos_kmer_y2, 0]            
        arrow_to_mat = Arrow(pdna_arrow, pmat, buff=0, color=RED)  
        self.add(arrow_to_mat)
        
        #pos_dna_rect = [get_wx_dna(0), 0, 0]
        #pmat = [get_wx_kmer(into_pos[0]), pos_kmer_y, 0]
        #rect_kmer.move_to(pos_dna_rect)

        for i in range(kmer_flanks,-kmer_flanks+len(dna_seq)):
            #Add to count
            mat_index = into_pos[i]
            num_mat[mat_index] = num_mat[mat_index] + 1

            #Get coordinates
            pos_dna_rect = [get_wx_dna(i), 0, 0]
            pdna = [get_wx_dna(i), pos_kmer_y1, 0]
            pmat = [get_wx_kmer(mat_index), pos_kmer_y2, 0]

            #Update graphics
            new_text = Text(str(num_mat[mat_index]), font_size=20).move_to(([get_wx_kmer(mat_index), pos_kmer_y, 0]))
            new_arrow_to_mat = Arrow(pdna, pmat, buff=0, color=RED)  #[0,0,0]
            self.play(
                rect_kmer.animate.move_to(pos_dna_rect),
                arrow_to_mat.animate.become(new_arrow_to_mat)#set_points_by_ends(pdna, pmat),
            )
            self.play(
                text_mat[mat_index].animate.become( new_text ),
            )
        
        self.wait()

Manim Community v0.20.1

In [56]:
%%manim -qm MustUseAngle


#####################################
##################################### Countsketch: need to use angles for comparison
#####################################

do_full_anim = True

def make_samples_for_genome_vec(x, rate_lambda):
    return([np.sign(y)*np.random.poisson(np.abs(y)*rate_lambda) for y in x ])
    
def rescale_gvec(x):
    scale_gvec = 1/100.0
    s = [y*scale_gvec for y in x]
    return(s)

class MustUseAngle(ThreeDScene):
    def construct(self):        
        axes = ThreeDAxes()
        self.set_camera_orientation(phi=20 * DEGREES, theta=-70 * DEGREES)
        self.add(axes)

        font_size_strain=30
        font_size_math=30

        color_samples = BLUE
        color_genomes = RED

        self.begin_ambient_camera_rotation(1*DEGREES, about='phi')

        ####### Text up left #############
        text_orig_c = Tex(r"Genome KMER counts: $C_i$",font_size=font_size_math, color=color_genomes)
        text_samp_c = Tex(r"Sequenced counts: $\tilde{C_i} \sim Poi(\lambda C_i) $; $\lambda$ is coverage", font_size=font_size_math, color=color_samples)
        text_samp_c.next_to(text_orig_c, DOWN)
        grp_ul = VGroup(text_orig_c, text_samp_c)
        self.add_fixed_in_frame_mobjects(grp_ul)
        grp_ul.to_corner(UL)
        text_orig_c.set_opacity(0)
        text_samp_c.set_opacity(0)
        
        ####### Genome vectors #############
        coord_len = 100.0
        coord_cell1 = [-coord_len, coord_len, 0.0]
        arrow_cell1 = Arrow(ORIGIN, rescale_gvec(coord_cell1), buff=0, color=color_genomes)
        text_cell1 = Tex('$C_1$: Bacillus', font_size=font_size_strain).next_to(arrow_cell1.get_end(), RIGHT)

        coord_len = 150.0
        coord_cell2 = [-coord_len, -coord_len, 0.0]
        arrow_cell2 = Arrow(ORIGIN, rescale_gvec(coord_cell2), buff=0, color=color_genomes)
        text_cell2 = Tex('$C_2$: Campylobacter', font_size=font_size_strain).next_to(arrow_cell2.get_end(), RIGHT)
        
        if do_full_anim:
            text_orig_c.set_opacity(1)
            self.play(Create(arrow_cell1))
            self.play(Create(text_cell1))
            self.play(Create(arrow_cell2))
            self.play(Create(text_cell2))
        else:            
            self.add(text_orig_c)
            self.add(arrow_cell1, text_cell1)
            self.add(arrow_cell2, text_cell2)
            
        if do_full_anim:
            self.wait()

        ####### Cells as samples #############    
        def add_samples(c):        
            #list_lambda = [ (i*2.0)/10.0 for i in range(10)]        
            list_lambda = [ (i*1.5)/10.0 for i in range(10)]        
            list_s = [make_samples_for_genome_vec(c, r) for r in list_lambda]
            #list_arrow = [Arrow(ORIGIN, rescale_gvec(s), buff=0, color=BLUE, stroke_width=1) for s in list_s]
            list_arrow = [Line(ORIGIN, rescale_gvec(s), buff=0, color=color_samples, stroke_width=3) for s in list_s]
            return(list_arrow)
        
        list_arrow1 = add_samples(coord_cell1)
        list_arrow2 = add_samples(coord_cell2)
        comb_list = list_arrow1+list_arrow2
        random.shuffle(comb_list)        
        
        if do_full_anim:
            self.play(text_samp_c.animate().set_opacity(1))
            for a in comb_list:
                self.play(Create(a, run_time=0.1))
        else:
            text_samp_c.set_opacity(1)
            for a in comb_list:
                self.add(a)

        if do_full_anim:
            self.wait()

        ####### Euclidian distance cannot be used #############
        p1 = list_arrow1[9].get_end()
        p2 = list_arrow2[9].get_end()
        b1 = BraceBetweenPoints(p1,p2)
        b1text = b1.get_text("Dissimilarity").scale(0.8)

        p1 = list_arrow1[2].get_end()
        p2 = list_arrow2[3].get_end()
        b2 = BraceBetweenPoints(p1,p2)
        b2text = b1.get_text("??")
        
        if do_full_anim:
            self.play(Create(b1), Create(b1text))
            self.wait()
            self.play(Create(b2), Create(b2text))
        else:
            self.add(b1, b1text)
            self.add(b2)
            
        if do_full_anim:
            self.wait()

        
        ####### Use angle instead #############
        self.remove(b1)
        self.remove(b1text)
        self.remove(b2)
        self.remove(b2text)
        
        # Create the angle arc
        angle_genomes = Angle(arrow_cell1, arrow_cell2, radius=0.5, color=YELLOW)
        angle_label = Tex(r"Similarity",font_size=font_size_math, color=YELLOW).next_to(angle_genomes, [-2,-1,0], buff=0.1)

        if do_full_anim:
            self.play(Create(angle_genomes), Create(angle_label))
        else:
            self.add(angle_genomes, angle_label)        
            
        if do_full_anim:
            self.wait()
        
        self.stop_ambient_camera_rotation(about='phi')

        ############## Move to 2 coordinates #############        
        phi, theta, focal_distance, gamma, distance_to_origin = self.camera.get_value_trackers()
        self.play(
            phi.animate.set_value(0*DEGREES),
            theta.animate.set_value(-90*DEGREES)
        )        
        if do_full_anim:
            self.wait()
        
        ############## Show cosine distance eq #############    
        text_cos1 = Tex(r"We can calculate the angle \\ using the scalar product",font_size=font_size_math, color=WHITE)
        text_cos2 = Tex(r"$Cos(\theta) = \frac{C_1 \cdot C_2}{\left\lVert C_1 \right\rVert   \left\lVert C_2 \right\rVert} $",
                        font_size=font_size_math, color=WHITE)
        text_cos3 = Tex(r"$u \cdot v = \sum_k u_i v_i $",
                        font_size=font_size_math, color=WHITE)
        self.add_fixed_in_frame_mobjects(text_cos1)
        text_cos1.move_to([3,3,0])
        text_cos2.move_to([3,2,0])
        text_cos3.move_to([3,1,0])

        self.play(Create(text_cos1))
        self.play(Create(text_cos2))
        self.wait()
        
        self.play(Create(text_cos3))
        self.wait()
        

Manim Community v0.20.1

In [662]:
%%manim -qm JaccardIsScalarProduct

#####################################
##################################### Countsketch: Jaccard index is a dot product
#####################################

do_full_anim = True

class JaccardIsScalarProduct(ThreeDScene):
    def construct(self):        

        ############### constants ##############
        pos_venn_y = 1

        
        ############### Show venn  ##############
        #self.camera.frame.scale(1 / 25)  
        mob = Circle(color=YELLOW, fill_opacity=0.5, stroke_width=0)#.scale(1)  
        mob2 = Circle(color=ORANGE, fill_opacity=0.5, stroke_width=0)#.scale(0.1)  
        venn = VGroup(mob, mob2).arrange(RIGHT, buff=-0.5).move_to([-3,pos_venn_y,0])

        txt_ani = Tex(r"ANI (average nucleotide identity) is \\ typically computed based on KMER overlap", font_size=30)
        txt_ani.move_to([0,3,0])

        txt_a = Text("A").move_to([-1-3,pos_venn_y,0])
        txt_b = Text("B").move_to([ 1-3,pos_venn_y,0])
        
        if do_full_anim:
            self.play(
                Create(venn),
                Create(txt_ani),
                Create(txt_a),
                Create(txt_b)
            )
            self.wait()
        else:
            self.add(venn)
            self.add(txt_ani)        
            self.add(txt_a)
            self.add(txt_b)
        

        ############ show jaccard ###############
        txt_jaccard = Tex(r"$J(A,B) = \frac{  \left\lVert A \cap B \right\rVert   }{    \left\lVert A \cup B \right\rVert   }$")
        txt_jaccard.move_to([3,pos_venn_y,0])

        txt_cap = Tex(r"$ \left\lVert A \cap B \right\rVert  =  \left\lVert \{ x \in A \land x \in B \} \right\rVert  $")
        txt_cap.move_to([3,pos_venn_y-1,0])
        
        if do_full_anim:
            self.play(Create(txt_jaccard))
            self.wait()
            self.play(Create(txt_cap))
            self.wait()
        else:
            self.add(txt_jaccard)
            self.add(txt_cap)

        
        ############### Show truth table  ##############

        table_logic = MobjectMatrix([
            [Tex(r"$\bot \land \top$"), Tex(r"$\bot$")],
            [Tex(r"$\top \land \bot$"), Tex(r"$\bot$")], 
            [Tex(r"$\bot \land \top$"), Tex(r"$\bot$")], 
            [Tex(r"$\top \land \top$"), Tex(r"$\top$")]
        ])
        table_logic.move_to([-5.2,-2,0]).scale(0.8)

        if do_full_anim:
            self.play(Create(table_logic))
            self.wait()
        else:
            self.add(table_logic)


        ############### Show mul table  ##############

        table_mul = MobjectMatrix([
            [Tex(r"$0 * 0$"), Tex(r"$0$")],
            [Tex(r"$1 * 0$"), Tex(r"$0$")], 
            [Tex(r"$0 * 1$"), Tex(r"$0$")], 
            [Tex(r"$1 * 1$"), Tex(r"$1$")]
        ])
        table_mul.move_to([-2,-2,0]).scale(0.8)

        if do_full_anim:
            self.play(Create(table_mul))
            self.wait()
        else:
            self.add(table_mul)


        ############### Show scalar prod  ##############        

        txt_cap_mul = Tex(r"$ \left\lVert A \cap B \right\rVert  =  \sum_i a_i b_i  $")
        txt_cap_mul.move_to([3,pos_venn_y-2,0])

        if do_full_anim:
            self.play(Create(txt_cap_mul))
            self.wait()
        else:
            self.add(txt_cap_mul)
        
        ############### Show conclusion  ##############        

        txt_conc = Tex(r"ANI can be calculated as \\ scalar product  if \\ KMERs are unique-ish!")
        txt_conc.move_to([3,pos_venn_y-4,0])
        
        if do_full_anim:
            self.play(Create(txt_conc))
            self.wait()
        else:
            self.add(txt_conc)

Manim Community v0.20.1

In [666]:
%%manim -qm HowManyDimensions

#####################################
##################################### Countsketch: There are too many dimensions
#####################################

do_full_anim = True

class HowManyDimensions(ThreeDScene):
    def construct(self):        

        txt_prob = Tex(r"Our space is really high-dimensional: 31 bases per KMER \\  $ \rightarrow 4^{31}$ dimensions!")
        txt_prob.move_to([0,2,0])

        if do_full_anim:
            self.play(Create(txt_prob))
            self.wait()
        else:
            self.add(txt_prob)

        txt_dec = Tex(r"$4^{31} \approx 10^{18}$ \\ $ \approx 1,000,000,000,000,000,000$ !")
        txt_dec.move_to([0,0,0])

        if do_full_anim:
            self.play(Create(txt_dec))
            self.wait()
        else:
            self.add(txt_prob)


        txt_mem = Tex(r"One cell requires 1 exabyte memory to store... \\ Workaround needed!")
        txt_mem.move_to([0,-2,0])

        if do_full_anim:
            self.play(Create(txt_mem))
            self.wait()
        else:
            self.add(txt_mem)

        self.wait()

Manim Community v0.20.1

In [57]:
%%manim -qm CSShowProjection

#####################################
##################################### Countsketch: Random projection
#####################################

do_full_anim = True

def scalarprod(u,v):
    return( u[0]*v[0] + u[1]*v[1] + u[2]*v[2] )

def veclen(u):
    return( np.sqrt( scalarprod(u,u) ) )

def proj(v,n):
    norm_n = veclen(n)
    alpha = scalarprod(v,n) / (norm_n*norm_n)
    proj_v = np.array(v) - alpha * np.array(n)
    return(proj_v)

def rescale_gvec(x):
    scale_gvec = 1/100.0
    s = [y*scale_gvec for y in x]
    return(s)


def make_3d_angle(a,b, radius=0.5):
    #this function is an approximation. need quaternions.
    #rescale input
    a=np.array(a)
    b=np.array(b)
    interpols = [i/10.0 for i in range(0,11)]    
    intermed_vec = [ b*i + a*(1-i) for i in interpols]
    points = [v * radius / veclen(v) for v in intermed_vec]

    mlines = []
    for i in range(10):
        mlines.append(Line(points[i], points[i+1], color=YELLOW))
    return(mlines)
    

class CSShowProjection(ThreeDScene):
    def construct(self):        

        ####### Constants #############

        font_size_strain=30
        font_size_math=30

        color_samples = BLUE
        color_genomes = RED
        color_proj = GREEN

        plane_vector=[1,0,1]

        coord_len = 100.0
        coord_cell1 = [-coord_len*2, coord_len, 0.0]
        coord_len = 150.0
        coord_cell2 = [-coord_len*2, -coord_len, 0.0]
        
        coord_cell1_proj = proj(coord_cell1, plane_vector) 
        coord_cell2_proj = proj(coord_cell2, plane_vector) 
        
        ####### Original coordinate system #############
        axes = ThreeDAxes()
        self.set_camera_orientation(phi=20 * DEGREES, theta=-70 * DEGREES)
        self.add(axes)


        self.begin_ambient_camera_rotation(30*DEGREES, about='phi') 

        ####### Text up left #############
        text_orig_c = Tex(r"Consider our original vectors: $C_i$ \\ Want to compute scalar products $C_i \cdot C_i$",font_size=font_size_math, color=color_genomes)
        text_proj_c = Tex(r"We compute projections $SC_i$ into random subspace S \\ Compute instead $SC_i \cdot SC_i$", font_size=font_size_math, color=color_proj)
        text_proj_c2 = Tex(r"Angle is almost retained, \\ but $SC_i$ is much smaller! (4096 dim)", font_size=font_size_math, color=color_proj)
        text_proj_c2.next_to(text_proj_c2, DOWN)
        grp_ul = VGroup(text_orig_c) #text_proj_c
        grp_ur = VGroup(text_proj_c, text_proj_c2)
        self.add_fixed_in_frame_mobjects(grp_ul)
        self.add_fixed_in_frame_mobjects(grp_ur)
        grp_ul.to_corner(UL)
        grp_ur.to_corner(UR)
        text_proj_c.set_opacity(0)
        text_proj_c2.set_opacity(0)

        
        ####### Genome vectors #############
        arrow_cell1 = Arrow(ORIGIN, rescale_gvec(coord_cell1), buff=0, color=color_genomes)
        #text_cell1 = Text('U', font_size=font_size_strain).next_to(arrow_cell1.get_end(), RIGHT)

        arrow_cell2 = Arrow(ORIGIN, rescale_gvec(coord_cell2), buff=0, color=color_genomes)
        #text_cell2 = Text('V', font_size=font_size_strain).next_to(arrow_cell2.get_end(), RIGHT)

        # Create the angle arc, orig vec
        angle_genomes = Angle(arrow_cell1, arrow_cell2, radius=0.5, color=YELLOW)
        #angle_label = Tex(r"$\theta$",font_size=font_size_math, color=YELLOW).next_to(angle_genomes, [-2,-1,0], buff=0.1)

        if do_full_anim:
            self.play(Create(arrow_cell1))
            #self.play(Create(text_cell1))
            self.play(Create(arrow_cell2))
            #self.play(Create(text_cell2))
            self.play(Create(angle_genomes))#, Create(angle_label))
            self.wait()
        else:
            self.add(angle_genomes)#, angle_label)        
            self.add(arrow_cell1)
            self.add(arrow_cell2)

        
        ####### Projection #############
        arrow_cell1_dash = DashedVMobject(Line(rescale_gvec(coord_cell1), rescale_gvec(coord_cell1_proj), buff=0, color=WHITE))
        arrow_cell2_dash = DashedVMobject(Line(rescale_gvec(coord_cell2), rescale_gvec(coord_cell2_proj), buff=0, color=WHITE))
        
        plane = NumberPlane()        
        plane.rotate( 45 * DEGREES, axis=UP).scale(0.5)

        text_proj_c.set_opacity(1)
        
        if do_full_anim:
            self.play(Create(plane))
            self.wait()
            self.play(Create(arrow_cell1_dash))
            self.play(Create(arrow_cell2_dash))
            self.wait()
        else:
            self.add(plane)
            self.add(arrow_cell1_dash)
            self.add(arrow_cell2_dash)
            
        
        ####### Projected vectors #############
        arrow_cell1_proj = Arrow(ORIGIN, rescale_gvec(coord_cell1_proj), buff=0, color=color_proj)
        arrow_cell2_proj = Arrow(ORIGIN, rescale_gvec(coord_cell2_proj), buff=0, color=color_proj)
        

        # Create the angle arc, projected vec. cannot use normal function
        angle_genomes_proj = make_3d_angle(rescale_gvec(coord_cell1_proj), rescale_gvec(coord_cell2_proj))
        #angle_label_proj = Tex(r"$\alpha$",font_size=font_size_math, color=YELLOW).next_to(angle_genomes, [-2,-1,0], buff=0.1)

        text_proj_c2.set_opacity(1)

        if do_full_anim:
            self.play(Create(arrow_cell1_proj))
            #self.play(Create(text_cell1))
            self.play(Create(arrow_cell2_proj))
            self.play(*[Create(x) for x in angle_genomes_proj])
            self.wait()
        else:
            self.add(arrow_cell1_proj)
            self.add(arrow_cell2_proj)
            self.add(*angle_genomes_proj)    

        if do_full_anim:
            self.play(Wait(run_time=3))
        self.stop_ambient_camera_rotation(about='phi')

Manim Community v0.20.1

In [624]:
%%manim -qm CSIsOrthogonal

#####################################
##################################### High dimensional vectors are almost always orthogonal
#####################################

do_full_anim = True

class CSIsOrthogonal(ThreeDScene):
    def construct(self):        

        vec1 = Matrix([[1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]],
            element_alignment_corner=UL,
            left_bracket="(",
            right_bracket=")").scale(0.4).move_to([0,3,0])

        vec2 = Matrix([[0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0]],
            element_alignment_corner=UL,
            left_bracket="(",
            right_bracket=")").scale(0.4)#.move_to([0,-1,0])

        txt1 = Tex(r"What is probably of a random vector pointing the same way?")
        txt2 = Tex(r"Probability is $ \sim 1/n$, n being number of dimensions")
        txt3 = Tex(r"$n=4^{31}$ means almost 0 chance!")

        txt1.next_to(vec1, DOWN)
        vec2.next_to(txt1, DOWN)
        txt2.next_to(vec2, DOWN)
        txt3.next_to(txt2, DOWN)
        
        if do_full_anim:
            self.play(Create(vec1))
            self.wait()
            self.play(Create(txt1))
            self.wait()
            self.play(Create(vec2))
            self.wait()
            self.play(Create(txt2))
            self.wait()
            self.play(Create(txt3))
            self.wait()
        else:
            self.add(vec1)
            self.add(vec2)
            self.add(txt1)
            self.add(txt2)
            self.add(txt3)
            

Manim Community v0.20.1

In [630]:
%%manim -qm CSgenerateImplicit

#####################################
##################################### Countsketch: How we merge data
#####################################

do_full_anim = True

def get_wx_dna(x):
    return(x*scale_dna_x - 5 )

def get_wx_kmer(x):
    return(x*scale_kmer_x - 5 )

class CSgenerateImplicit(ThreeDScene):
    def construct(self):        

        font_size_math=30

        
        txt1 = Tex(r"How do we design random matrix S?", font_size=font_size_math)
        txt2 = Tex(r"We can use two hash functions taking KMERs as input", font_size=font_size_math)  #: \\ $g(k)$ and $h(k)$
        txt3 = Tex(r"$g(k)$ maps to $\{-1,+1\}$", font_size=font_size_math)
        txt4 = Tex(r"$h(k)$ maps to $[1 ... N]$, \\where N is number of random dimensions", font_size=font_size_math)
        txt5 = Tex(r"Define matrix: $S_{ij}=\sum_j g(j) 1_{h(j)=i} $", font_size=font_size_math)
        txt6 = Tex(r"We can use an implicit definition of $S$\\ to directly generate projected KMER counts $Su$", font_size=font_size_math)
        
        txt1.move_to([0,3.5,0])
        txt2.next_to(txt1, DOWN, buff=0.5)

        txt3.move_to([-3,1.5,0])
        txt4.next_to(txt3, DOWN, buff=0.5)

        txt5.move_to([3.2,1,0])
        txt6.move_to([0,-1,0])

        rect_eq = Rectangle(width=12, height=1.8).move_to([0,0.9,0])
        
        if do_full_anim:
            self.play(Create(txt1))
            self.wait()
            self.play(Create(txt2))
            self.wait()
            self.play(Create(txt3))
            self.wait()
            self.play(Create(txt4))
            self.wait()            
            self.play(Create(txt5))
            self.play(Create(rect_eq))            
            self.wait()
            self.play(Create(txt6))
            self.wait()
        else:
            self.add(txt1)
            self.add(txt2)
            self.add(txt3)
            self.add(txt4)
            self.add(txt5)
            self.add(txt6)
            self.add(rect_eq)


        ############################ Simulate counting #################################

        ######### constants #################
        scale_dna_x = 0.3
        scale_kmer_x = 0.25
        window_size = scale_dna_x*5
        kmer_size = 5
        kmer_flanks = int(kmer_size/2)
        height_kmer_rect = 0.7
        pos_dna = -2
        pos_kmer_y = -3.5
        pos_kmer_y1 = pos_dna - height_kmer_rect/2.0
        pos_kmer_y2 = pos_kmer_y + 0.2
        num_subdim=3
        
        dna_seq = "AACGACAGTACCGA"#CATCGACACTACTACG"
        
        rng = np.random.default_rng(seed=1)
        into_pos = [rng.integers(0,num_subdim) for i in range(len(dna_seq))]
        
        into_sign = [rng.integers(0,2)*2-1 for i in range(len(dna_seq))]
        
        ######### Show count vector #################        

        ## Place DNA
        for i in range(len(dna_seq)):
            text_dna = Text(dna_seq[i], font_size=20).move_to(([get_wx_dna(i), pos_dna, 0]))
            self.add(text_dna)

        self.wait()
        
        ## Place matrix
        num_mat = [0 for i in range(num_subdim)]
        text_mat = [Text("0", font_size=15).move_to([get_wx_kmer(i), pos_kmer_y, 0]) for i in range(len(num_mat))]
        for t in text_mat:
            self.add(t)
            
        self.wait()
        
        ######### animate the counting #################

        ## Prep rect                 
        pos_dna_rect = [-20, pos_dna, 0] # [get_wx_dna(0), pos_dna, 0]
        rect_kmer = Rectangle(width=window_size, height=height_kmer_rect).move_to(pos_dna_rect)
        self.add(rect_kmer)
        #self.play(Create(rect_kmer))
        
        ## Prep arrow  
        pdna_arrow = [get_wx_dna(0), pos_kmer_y1, 0]
        pmat = [get_wx_kmer(into_pos[0]), pos_kmer_y2, 0]            
        arrow_to_mat = Arrow(pdna_arrow, pmat, buff=0, color=RED)  
        self.add(arrow_to_mat)
        
        for i in range(kmer_flanks,-kmer_flanks+len(dna_seq)):
            #Add to count
            mat_index = into_pos[i]
            num_mat[mat_index] = num_mat[mat_index] + into_sign[i]
            
            #Get coordinates
            pos_dna_rect = [get_wx_dna(i), pos_dna, 0]
            pdna = [get_wx_dna(i), pos_kmer_y1, 0]
            pmat = [get_wx_kmer(mat_index), pos_kmer_y2, 0]

            #Update graphics
            new_text = Text(str(num_mat[mat_index]), font_size=20).move_to(([get_wx_kmer(mat_index), pos_kmer_y, 0]))
            new_arrow_to_mat = Arrow(pdna, pmat, buff=0, color=RED)  #[0,0,0]
            self.play(
                rect_kmer.animate.move_to(pos_dna_rect),
                arrow_to_mat.animate.become(new_arrow_to_mat)#set_points_by_ends(pdna, pmat),
            )
            self.play(
                text_mat[mat_index].animate.become( new_text ),
            )
        
        self.wait()

Manim Community v0.20.1

In [90]:
%%manim -qm NeedKNN

do_full_anim = True

#####################################
##################################### Countsketch: kNN
#####################################


########### constants #############
pos_mat_x = 3
pos_mat_y = -1

pos_graph_x = 4
pos_graph_y = 2.5

font_size_math=30


########### Generate points for two clusters #############
rng = np.random.default_rng(seed=1)
list_point = [[
    rng.uniform(-3, 3),
    rng.uniform(-1, 1),
    0] for i in range(30) ]  #30

def close_to_circ1(xy):
    x=xy[0]
    y=xy[1]
    dx = x-2
    dy = y
    return(dx*dx + dy*dy < 1)

def close_to_circ2(xy):
    x=xy[0]
    y=xy[1]
    dx = x+2
    dy = y
    return(dx*dx + dy*dy < 1)


list_point = [p for p in list_point if close_to_circ1(p) or close_to_circ2(p)]
list_point = [
    [p[0]+pos_graph_x, p[1]+pos_graph_y, 0] 
    for p in list_point
]  #move all points

######### Make distance matrix, made up numbers 
matrix_dense = [
    [
        int(rng.integers(0,5))
        for i in range(len(list_point))  ### TODO make symmetric
    ]
    for j in range(len(list_point))
]
for i in range(len(matrix_dense)):
    matrix_dense[i][i] = 0


################ construct knn matrix
from sklearn.neighbors import NearestNeighbors
import numpy as np
X = np.array(list_point) 
nbrs = NearestNeighbors(n_neighbors=4, algorithm='ball_tree').fit(X)
knn_distances, knn_indices = nbrs.kneighbors(X)
distmat_knn = nbrs.kneighbors_graph(X).toarray()



######### Make sparse matrix
matrix_sparse = [
    [
        ""
        for i in range(len(list_point))
    ]
    for j in range(len(list_point))
]
for i in range(len(matrix_dense)):
    matrix_sparse[i][i] = "0"


for one_nei in knn_indices:
    pi = one_nei[0]
    for pj in one_nei[1:]:
        matrix_sparse[pi][pj] = str(matrix_dense[pi][pj])
        matrix_sparse[pj][pi] = str(matrix_dense[pj][pi])

def matrix_to_text(mat):
    return(list(map(
        lambda sublist: list(map(lambda x: Text(str(x)), sublist)), 
        mat
    )))


class NeedKNN(ThreeDScene):
    def construct(self):        
        
        ############ Layout matrices
        if True:
            txt_matrix_dense = matrix_to_text(matrix_dense)
            elem_mat_dense = MobjectMatrix(
                txt_matrix_dense,
                element_alignment_corner=UL,
                left_bracket="(",
                right_bracket=")").scale(0.25).move_to([pos_mat_x,pos_mat_y,0])
            elem_mat_dense.set_opacity(0)
            
        if True:
            txt_matrix_sparse = matrix_to_text(matrix_sparse)
            elem_mat_sparse = MobjectMatrix(
                txt_matrix_sparse,
                element_alignment_corner=UL,
                left_bracket="(",
                right_bracket=")").scale(0.25).move_to([pos_mat_x,pos_mat_y,0])
            elem_mat_sparse.set_opacity(1)

        ############ Layout graph
        graph_lines_all = []
        for p in list_point:
            for q in list_point:
                graph_lines_all.append(Line(p,q, color=GRAY))

        graph_lines_knn = []
        for one_nei in knn_indices:
            i = one_nei[0]
            for j in one_nei[1:]:            
                graph_lines_knn.append(Line(list_point[i],list_point[j], color=GREEN))

        graph_points = []
        for p in list_point:
            op = Circle(0.05, color=WHITE, fill_opacity=1,z_index=10)
            op.move_to(p)
            graph_points.append(op)
            #graph_points.append(Point(p, color=WHITE)) 

        ############ Intro
        txt_question = Tex(r"How do we find the most similar genomes?", font_size=font_size_math)
        txt_tree1 = Tex(r"Most algorithms require all \\ pairwise distances to be computed", font_size=font_size_math)
        txt_tree2 = Tex(r"Our vector representation enable \\ computation of K nearest neighbours", font_size=font_size_math)
        txt_tree3 = Tex(r"This drastically reduces computing \\ requirements for many cells", font_size=font_size_math)
        txt_question.move_to([-3,3,0])
        txt_tree1.next_to(txt_question, [0,-1,0])
        txt_tree2.next_to(txt_tree1, [0,-1,0])
        txt_tree3.next_to(txt_tree2, [0,-1,0])
        
        ############ Show genomes unconnected ###############
        if do_full_anim:
            self.play(Create(txt_question))
            self.play(
                *map(Create, graph_points)
            )
            self.wait()
        else:
            self.add(txt_question)
            self.add(txt_tree)
            self.add(*graph_points)
                
        ############ Show full graph ###############
        if do_full_anim:
            self.play(Create(txt_tree1))
            self.play(
                *map(Create, graph_lines_all)
            )
            self.play(
                elem_mat_dense.animate.set_opacity(1)
            )
            self.wait()
        else:
            self.add(*graph_lines_knn)
        
        
        ############ Show knn graph ###############
        if do_full_anim:
            self.play(Create(txt_tree2))
            self.play(
                *map(Create,graph_lines_knn)       
            )
            self.wait()
            self.play(
                elem_mat_dense.animate.set_opacity(0),
                elem_mat_sparse.animate.set_opacity(1),
            )
            self.play(Create(txt_tree3))
            self.wait()
        else:
            elem_mat_dense.set_opacity(1)


Manim Community v0.20.1

[03/08/26 21:02:20] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=17400;file:///home/mahogny/miniconda3/lib/python3.12/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=538683;file:///home/mahogny/miniconda3/lib/python3.12/site-packages/manim/utils/hashing.py#161\161]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

                    WARNING  You can disable this warning by setting disable_caching_warning to True ]8;id=841092;file:///home/mahogny/miniconda3/lib/python3.12/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=358726;file:///home/mahogny/miniconda3/lib/python3.12/site-packages/manim/utils/hashing.py#167\167]8;;\
                             in your config file.                                                                  

[03/08/26 21:02:30] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=184328;file:///home/mahogny/miniconda3/lib/python3.12/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=892625;file:///home/mahogny/miniconda3/lib/python3.12/site-packages/manim/utils/hashing.py#161\161]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

                    WARNING  You can disable this warning by setting disable_caching_warning to True ]8;id=126157;file:///home/mahogny/miniconda3/lib/python3.12/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=992987;file:///home/mahogny/miniconda3/lib/python3.12/site-packages/manim/utils/hashing.py#167\167]8;;\
                             in your config file.                                                                  

[03/08/26 21:02:40] WARNING  It looks like the scene contains a lot of sub-mobjects. Caching is      ]8;id=144831;file:///home/mahogny/miniconda3/lib/python3.12/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=222576;file:///home/mahogny/miniconda3/lib/python3.12/site-packages/manim/utils/hashing.py#161\161]8;;\
                             sometimes not suited to handle such large scenes, you might consider                  
                             disabling caching with --disable_caching to potentially speed up the                  
                             rendering process.                                                                    

                    WARNING  You can disable this warning by setting disable_caching_warning to True ]8;id=100028;file:///home/mahogny/miniconda3/lib/python3.12/site-packages/manim/utils/hashing.py\hashing.py]8;;\:]8;id=416736;file:///home/mahogny/miniconda3/lib/python3.12/site-packages/manim/utils/hashing.py#167\167]8;;\
                             in your config file.                                                                  

In [73]:
%%manim -qm LinkNN

#####################################
##################################### Countsketch: Link to neural networks
#####################################

do_full_anim = True

node_dx = 4
node_dy = 2
num_layer = 3
num_nodes_in_layer = 3
node_radius = 0.25

def make_nn_pos(layer,i):
    return([
        (layer-num_layer/2+0.5)*node_dx, 
        (i-num_nodes_in_layer/2+0.5)*node_dy, 
        0])

class LinkNN(MovingCameraScene):
    def construct(self):        

        color_node = WHITE
        color_edge = GRAY_C
        
        #### show neural network #############
        all_nodes = []
        for cur_layer in range(num_layer):
            for cur_node in range(num_nodes_in_layer):
                all_nodes.append(Circle(node_radius,color=color_node).move_to(make_nn_pos(cur_layer, cur_node)))
        
        all_edges = []
        for cur_layer in range(1,num_layer):
            for cur_node_to in range(num_nodes_in_layer):
                for cur_node_from in range(num_nodes_in_layer):
                    pos_from = make_nn_pos(cur_layer-1, cur_node_from)
                    pos_to   = make_nn_pos(cur_layer,   cur_node_to)
                    all_edges.append(Arrow(pos_from, pos_to, color=color_edge, stroke_width=1, tip_length=0.1).scale(0.95))

        if do_full_anim:
            self.play(
                *[Create(n) for n in all_nodes]
            )
            self.play(
                *[Create(n) for n in all_edges],
            )
            self.wait()
        else:
            self.add(*all_nodes)
            self.add(*all_edges)

        #### zoom in on one node #############
        txt_f = Tex(r"$f(x)$", font_size=20)
        self.play(
            self.camera.frame.animate.scale(0.3).move_to([0,-0.5,0]),
            Create(txt_f)
        )
        if do_full_anim:
            self.wait()
        
        #### show eq #############
        txt_eq = Tex(r"$f(x)=x \cdot w + b$", font_size=20).move_to([0,-0.5,0])
        if do_full_anim:
            self.play(Create(txt_eq))
            self.wait()
        else:
            self.add(txt_eq)

        #### show gpu #############
        txt_gpu = Tex(r"Neural network are all scalar products! \\ Our algorithm fits well on GPUs", font_size=20).next_to(txt_eq,[0,-1,0])
        #move_to([0,-0.5,0])
        img_gpu = ImageMobject("img/gpu_geforce.png").move_to([1.2,0,0]).scale(0.3)
        img_gpu.set_opacity(0)
        if do_full_anim:
            self.play(
                img_gpu.animate.set_opacity(1),
                Create(txt_gpu)
            )
            self.wait()
        else:
            img_gpu.set_opacity(1)
            self.add(txt_gpu)
        self.wait()



Manim Community v0.20.1